In [0]:
from pyspark.sql.functions import current_timestamp, lit
import yaml

In [0]:
with open("/Workspace/Users/kahledtrojan@gmail.com/databricks_lakehouse_pipeline_de/bike_lakehouse/config.yaml","r") as f:
    config = yaml.safe_load(f)

In [0]:
crm_tables = ["cust_info","prd_info","sales_details"]
erp_tables = ["CUST_AZ12","LOC_A101","PX_CAT_G1V2"]





def load_to_bronze(tables,source_system):
    for table in tables:
        df = (
            spark.read.option("header","true")
            .option("inferSchema","true")
            .csv(f"/Volumes/bike_lakehouse/bronze/source_systems/{table}.csv")
        )

        df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

        df.write.mode("overwrite").saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")



for source_system,details in config["source_systems"].items():

    load_to_bronze(details["tables"],source_system)




In [0]:
from pyspark.sql.functions import current_timestamp, lit
import yaml


with open("/Workspace/Users/kahledtrojan@gmail.com/databricks_lakehouse_pipeline_de/bike_lakehouse/config.yaml","r") as f:
    config = yaml.safe_load(f)


def load_to_bronze(tables,source_system):
    for table in tables:
        df = (
            spark.read.option("header","true")
            .option("inferSchema","true")
            .csv(f"/Volumes/bike_lakehouse/bronze/source_systems/{table}.csv")
        )

        df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

        df.write.mode("overwrite").saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")



for source_system,details in config["source_systems"].items():

    load_to_bronze(details["tables"],source_system)

In [0]:
%sql
DROP TABLE IF EXISTS bike_lakehouse.bronze.crm_cust_info;
DROP TABLE IF EXISTS  bike_lakehouse.bronze.crm_prd_info;
DROP TABLE IF EXISTS  bike_lakehouse.bronze.crm_sales_details;
DROP TABLE IF EXISTS  bike_lakehouse.bronze.erp_CUST_AZ12;
DROP TABLE IF EXISTS bike_lakehouse.bronze.erp_LOC_A101;
DROP TABLE IF EXISTS bike_lakehouse.bronze.erp_PX_CAT_G1V2;


In [0]:
%sql
SELECT * 
FROM bike_lakehouse.bronze.crm_cust_info

In [0]:
with open("/Workspace/Users/kahledtrojan@gmail.com/databricks_lakehouse_pipeline_de/bike_lakehouse/config.yaml","r") as f:
    config = yaml.safe_load(f)

In [0]:
jdbc_url = (
    "jdbc:postgresql://ep-rapid-darkness-a5qejnez-pooler.us-east-2.aws.neon.tech:5432/neondb"
)

properties = {
    "user": "neondb_owner",
    "password": "npg_MU9f0FxKVeDo",
    "driver": "org.postgresql.Driver"
}
def load_to_bronze(tables,source_system):
    for table in tables:

        df = (
            spark.read
            .jdbc(
                url=jdbc_url,
                table=F"{source_system}_{table}",
                properties=properties
            )
        )

        df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

        df.write.mode("overwrite").saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")


for source_system,details in config["source_systems"].items():

    load_to_bronze(details["tables"],source_system)
